### Training Data Generation

This section builds a pool of raw labels and logits from multiple LLMs to train MoLAM.

What it does:
- Samples up to 10,000 items from the train split of the chosen dataset (or fewer if the dataset is smaller).
- Prompts an LLM 10 times per item to obtain multiple label candidates (captures agreement/uncertainty).
- Extracts per-label scores (logits mapped to the dataset’s label space) from the LLM’s last-token distribution.
- Saves one JSON line per example to `output_exp/molam_train_raw_data/{dataset}_{llm}.json` with:
  - `label`: ground-truth label text (lowercased)
  - `llm_logits`: list[float] of length C (C = number of labels)
  - `llm_label`: list[str] of length 10 (labels across 10 generations)
  - `text`: original input text

Inputs:
- `dataset`: one of ["ag_news", "imdb", "trec", "pubmed-20k-rct"]
- `llm_model`: one of ["gemma", "llama", "mistral", "qwen", "yi"]

Key settings:
- Device: `cuda:0`
- Generation: `do_sample=True`, `temperature=0.7`, `top_p=0.9`, `max_new_tokens=800`
- Label set: `setup.get_label_list(dataset)`

Notes:
- This step is GPU-intensive (24G CUDA memory GPU is required); adjust the generation parameters in `generate_output` if needed.
- Ensure you have access to the selected model checkpoints and are authenticated where required.

In [ ]:
import os
import json

from tqdm import tqdm
from torch.utils.data import Subset

from labelGen import extract_output, get_llm_logit
from setup import get_llm_model, get_label_list, get_prompt, load_data, get_item

In [ ]:
def generate_output(input_ids, llm_model, llm_tokenizer):
    return llm_model.generate(input_ids,
                              max_new_tokens=800,
                              do_sample=True,
                              temperature=0.7,
                              top_p = 0.9,
                              pad_token_id=llm_tokenizer.eos_token_id,
                              eos_token_id=llm_tokenizer.eos_token_id
                              )

In [ ]:
def get_MoLAM_train_data(dataset, llm_model):
    os.makedirs(f"output_exp/molam_train_raw_data", exist_ok=True)
    
    ds = load_data(dataset)["train"].shuffle(666)
    ds_labels = get_label_list(dataset)
    label_list = get_label_list(dataset)
    model, tokenizer = get_llm_model("cuda:0", llm_model)

    n_sample = 10000 if len(ds) >= 10000 else len(ds)  # set the sample pool for training data; this is the sample pool size not the final training data size
    sampled_data = Subset(ds, list(range(n_sample)))

    for item in tqdm(sampled_data):
        t_text, t_label = get_item(dataset, item)
   
        messages = [{"role": "user", "content": get_prompt(dataset, article=t_text)}]
        input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to("cuda:0")

        llm_labels = []

        for j in range(10):
            outputs = generate_output(input_ids, model, tokenizer)
            llm_labels.append(extract_output(llm_model, tokenizer.decode(outputs[0])))

        llm_logits = get_llm_logit(model, tokenizer, input_ids, label_list)
        
        output_f = open(f"output_exp/molam_train_raw_data/{dataset}_{llm_model}.json", "a")
        output_f.write(json.dumps({"label": ds_labels[t_label].lower(),
                                   "llm_logits": llm_logits,
                                   "llm_label": llm_labels,
                                   "text": t_text})+"\n") 
        output_f.close()

In [ ]:
for dataset in ["ag_news", "imdb", "trec", "pubmed-20k-rct"]:
    for llm_model in ["gemma", "llama", "mistral", "qwen", "yi"]:
        get_MoLAM_train_data(dataset, llm_model)

### Training Data Formalization

This step merges raw outputs from multiple LLMs into a fixed-size meta-feature dataset used to train MoLAM.

Expected inputs:
- For each dataset and each model in ["gemma", "llama", "mistral", "qwen", "yi"], a file at
  `output_exp/molam_train_raw_data/{dataset}_{model}.json` created in the previous section.

What it builds:
- For each example and for each model, we concatenate:
  - `llm_logits` (length C)
  - Majority-vote distribution over the 10 generated labels (length C), computed by `output2major(...)`
- Overall meta feature per example: shape `(num_models, 2, C)` and later flattened to `(num_models * 2 * C)` for training.
- Targets (`meta_y`): one-hot true label of length C.

Outputs:
- `output_exp/molam_train_data/{dataset}_meta_x.npy`  # `(N, num_models, 2, C)` before flattening
- `output_exp/molam_train_data/{dataset}_meta_y.npy`  # `(N, C)`

Tip:
- If your raw files live under `output_exp/molam_train_raw_data`, ensure the `dir_raw` path in this section points there before running.

In [ ]:
import os
import json
import numpy as np

from setup import get_label_list

In [ ]:
def read_file(file_in):
    f = open(file_in, "r")
    lines = f.readlines()
    f.close()
    return lines

def output2major(label_list, output):
    return [output.count(label)/len(output) for label in label_list]

In [ ]:
def gen_meta_data(data):
    model_ls = ["gemma", "llama", "mistral", "qwen", "yi"]

    dir_raw="output_exp/molam/{data}_{model}.json"

    lines_raw = [read_file(dir_raw.format(model=model, data=data)) for model in model_ls]
    label_list = get_label_list(data)

    meta_x = []
    meta_y = []

    for i in range(len(lines_raw[0])):
        this_x = []

        for j in range(len(model_ls)):
            this_line = json.loads(lines_raw[j][i])
            
            label = this_line["label"]
            if j == 0: meta_y.append([1 if k==label_list.index(label) else 0 for k in range(len(label_list))])
            this_x.append(this_line["llm_logits"])
            this_x.append(output2major(label_list, this_line["llm_label"]))
            
        meta_x.append(this_x)
    
    np.save(f"output_exp/molam_train_data/{data}_meta_x.npy", meta_x)
    np.save(f"output_exp/molam_train_data/{data}_meta_y.npy", meta_y)

    return

In [ ]:
data_ls = ["ag_news", "imdb", "trec", "pubmed-20k-rct"]

for data in data_ls:
    print(f"Generating meta data for {data}...")
    gen_meta_data(data)

### MoLAM Training

Train MoLAM the meta features using iterative pseudo-labeling.

Procedure (per dataset):
- Randomly select 50 labeled examples; treat the rest as unlabeled.
- Iterate up to 100 pseudo-labeling steps:
  - Fit `xgboost.XGBRegressor(objective='binary:logistic')` on the current labeled set.
  - Predict probabilities for the unlabeled pool.
  - Select confident predictions where `max(prob) > pseudo_thresh`, convert to one-hot, and add to the labeled set.
- Repeat 10 times with different seeds and track the best accuracy. The best model is saved.

Hyperparameters:
- `pseudo_thresh`: confidence threshold for adding pseudo labels
- `lr` (learning_rate), `md` (max_depth), `ne` (n_estimators)
Recommended settings (see cell below)

Artifacts:
- Saved model: `output_exp/molam_model/molam_{dataset}.json`
- Console output: best accuracy per run and overall best accuracy.

In [ ]:
import random
import numpy as np
import xgboost as xgb

In [ ]:
def get_acc(y_test, y_pred):
    acc = 0
    for i in range(len(y_test)):
        acc += np.argmax(y_test[i])==np.argmax(y_pred[i])
    acc /= len(y_test)
    return acc

In [ ]:
def train_MoLAM(data, pseudo_thresh, lr, md, ne):
    os.makedirs(f"output_exp/molam_model", exist_ok=True)
    X = np.load(f"output_exp/molam_train_data/{data}_meta_x.npy")
    y = np.load(f"output_exp/molam_train_data/{data}_meta_y.npy")
    X = X.reshape(X.shape[0], X.shape[1]*X.shape[2])

    all_best_acc = -1

    for n in range(10): 
        this_best_acc = -1
        i_labeled = random.sample(list(range(X.shape[0])), 50)
        i_unlabeled = np.ones(len(X), dtype=bool)
        i_unlabeled[i_labeled] = False
        X_label, X_unlabel, y_label, y_unlabel = X[i_labeled], X[i_unlabeled], y[i_labeled], y[i_unlabeled]
        X_test, y_test = X[i_unlabeled], y[i_unlabeled]

        for j in range(100):            # Pseudo labeling iteration training
            xgb_regressor = xgb.XGBRegressor(objective='binary:logistic', 
                                                n_estimators=ne, 
                                                learning_rate=lr,
                                                max_depth=md,
                                                random_state=np.random.randint(10000))

            xgb_regressor.fit(X_label, y_label)
            y_pred_unlabel = xgb_regressor.predict(X_unlabel)
            y_pred_test = xgb_regressor.predict(X_test)

            acc = get_acc(y_test, y_pred_test)
            if acc > this_best_acc: this_best_acc = acc
            if acc > all_best_acc: 
                all_best_acc = acc
                xgb_regressor.save_model(f"output_exp/molam_model/molam_{data}.json")                
        
            i_selected = list(np.where(np.max(y_pred_unlabel, axis=1) > pseudo_thresh)[0])
            i_unselected = list(np.where(np.max(y_pred_unlabel, axis=1) <= pseudo_thresh)[0])
            if len(i_selected) == 0 or len(i_unselected) == 0:
                break
            X_pseudo, y_pseudo, X_unlabel, y_unlabel = X_unlabel[i_selected], y_pred_unlabel[i_selected], X_unlabel[i_unselected], y_unlabel[i_unselected]
            y_pseudo = (y_pred_unlabel[i_selected] == np.max(y_pred_unlabel[i_selected], axis=1, keepdims=True)).astype(int)
            y_label = np.concatenate([y_label, y_pseudo])
            X_label = np.concatenate([X_label, X_pseudo])
        
        print(f"Best acc for run {n}: {this_best_acc:8.6}")
    print(f"Overall best acc for {data}: {all_best_acc:8.6}")

In [ ]:
"""
    To train the MoLAM, use the following command
    Parameter for different datasets:
        ag_news:    pseudo_thresh=0.9, lr=0.07, md=5, ne=300
        imdb:       pseudo_thresh=0.9, lr=0.01, md=5, ne=300
        trec:       pseudo_thresh=0.9, lr=0.05, md=6, ne=300
        PubMed:     pseudo_thresh=0.9, lr=0.01, md=3, ne=500
"""

train_MoLAM("ag_news", pseudo_thresh=0.9, lr=0.07, md=5, ne=300)

### Demo for MoLAM Evaluation

A quick sanity check that loads a saved model and evaluates on the meta features used during training.

What it does:
- Load `output_exp/molam_model/molam_{dataset}.json`.
- Load `{dataset}_meta_x.npy` and `{dataset}_meta_y.npy`, flatten features to `(N, num_models * 2 * C)`, predict, and print accuracy via argmax.

Note:
- This is an in-sample evaluation intended for a quick check.

In [ ]:
import numpy as np
import xgboost as xgb

In [ ]:
def loadXGBmode(data):
    model = xgb.XGBRegressor()
    model.load_model(f'output_exp/molam_model/molam_{data}.json')
    return model

In [ ]:
def testXGBmodel(data):
    X = np.load(f"output_exp/molam_train_data/{data}_meta_x.npy")
    y = np.load(f"output_exp/molam_train_data/{data}_meta_y.npy")

    X = X.reshape(X.shape[0], X.shape[1]*X.shape[2])

    xgbModel = loadXGBmode(data)
    y_pred = xgbModel.predict(X)

    acc = get_acc(y, y_pred)
    print(f"{data}: {acc}")

In [ ]:
data_ls = ["ag_news", "imdb", "trec", "pubmed-20k-rct"]
for data in data_ls:
    testXGBmodel(data)